# NB01 · Chronos-2 zero-shot forecasting features  (**GPU T4 — the only GPU notebook** · budget-driven, ≤ ~3 h)

Turns the observed (faulted, delayed) streams into **calibrated-by-construction quantile forecasts** of six driver indicators per zone, 6→72 h ahead, which feed the risk head (NB04) as `chronos` features.

* Model: `amazon/chronos-2` (120 M, encoder-only, group attention). **Zones of one indicator are one multivariate task**, so group attention cross-learns across zones. Known-future covariates = the *issued* rain/temperature forecasts (with lead-dependent error), past covariates = observed rain/temperature. Never the realised future.
* Refreshed every 6 h; NB04 uses the latest refresh (staleness is a feature → train/serve parity).
* **Benchmark first**, then the notebook converts your GPU-hour budget into an episode quota (val/test/heldout/drift first, train with the remainder). Per-episode checkpoints → fully resumable.
* **Metrics that matter:** MAE & mean pinball loss vs persistence / seasonal-naive baselines (skill), 80 % interval coverage per horizon and per fault profile, throughput (series/s), by indicator & hazard.
* LoRA fine-tuning is intentionally **not** included (first item on the cut list in the blueprint); zero-shot is the default.

**Settings:** Accelerator = GPU T4 · Internet = ON (for `pip install chronos-forecasting` and the HF weights; or attach them as datasets).
**Inputs:** NB00 output.  **Outputs:** `chronos_feats/<split>/<ep>.npz`, `chronos_metrics.csv`, `chronos_info.json`.

In [ ]:
import subprocess, sys
try:
    from chronos import Chronos2Pipeline
except Exception:
    print(subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'chronos-forecasting>=2.0'], capture_output=True, text=True).stdout[-300:])

In [ ]:
import os, sys, glob, json, time, math, warnings, pickle
import numpy as np, pandas as pd
import matplotlib; import matplotlib.pyplot as plt
from tqdm.auto import tqdm
warnings.filterwarnings('ignore'); plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .25, 'figure.figsize': (9, 4)})
ON_KAGGLE = os.path.exists('/kaggle/working')
IN_ROOT = '/kaggle/input' if ON_KAGGLE else os.environ.get('W72_IN', './inputs')
OUT = '/kaggle/working' if ON_KAGGLE else os.environ.get('W72_OUT', './outputs')
FIG = f'{OUT}/figs'; os.makedirs(FIG, exist_ok=True)
def find(name, required=True):
    hits = sorted(glob.glob(f'{IN_ROOT}/**/{name}', recursive=True) + glob.glob(f'{OUT}/**/{name}', recursive=True))
    if not hits and required: raise FileNotFoundError(f'{name} not found under {IN_ROOT} - add the producing notebook/dataset via "Add Input"')
    return hits[0] if hits else None
def savefig(n): plt.tight_layout(); plt.savefig(f'{FIG}/{n}.png', bbox_inches='tight'); plt.show()
sys.path.insert(0, os.path.dirname(find('w72_common.py'))); import w72_common as W

In [ ]:
QUICK = False
CFG = dict(model='amazon/chronos-2', ctx=168, pred=72, budget_h=2.5, batch=16, quota=dict(val=1.0, test=1.0, heldout=1.0, drift=1.0, train=1.0), max_eps=dict(val=100, test=100, heldout=100, drift=40, train=500))
if QUICK: CFG.update(budget_h=0.05, max_eps=dict(val=3, test=3, heldout=3, drift=2, train=6), batch=8)
CTX, PRED = CFG['ctx'], CFG['pred']; T = 240; REF_T = np.arange(48, 168, 6)          # refresh times: need t+72 <= 239 for truth
TGT = [W.VI[n] for n in W.CH_TARGETS]; HS = [6, 24, 72]; QL = [0.1, 0.5, 0.9]; HEV = [1, 3, 6, 12, 24, 48, 72]
MAN = pd.read_csv(find('w72_manifest.csv')); ROOT = os.path.dirname(find('w72_manifest.csv')); CITY = {'anchor': W.make_city('anchor'), 'heldout': W.make_city('heldout')}
CF = f'{OUT}/chronos_feats'; [os.makedirs(f'{CF}/{s}', exist_ok=True) for s in ['train', 'val', 'test', 'heldout', 'drift']]

In [ ]:
# ---------------- pipeline loading (real Chronos-2 on GPU, or a loudly-flagged naive fallback so the notebook still completes)
class NaiveQuantilePipeline:
    """FALLBACK ONLY: persistence + sqrt-horizon Gaussian band, same predict_quantiles layout (n_var, pred_len, n_q). Not a foundation model."""
    def predict_quantiles(self, inputs, prediction_length, quantile_levels, **kw):
        from scipy.stats import norm
        outs = []
        for d in inputs:
            y = np.asarray(d['target'], np.float32); last = np.array([r[~np.isnan(r)][-1] if (~np.isnan(r)).any() else 0.0 for r in y])
            sd = np.array([np.nanstd(np.diff(r)) if np.isfinite(np.nanstd(np.diff(r))) else 0.1 for r in y]) + 1e-3
            h = np.arange(1, prediction_length + 1); z = norm.ppf(quantile_levels)
            outs.append(last[:, None, None] + sd[:, None, None] * np.sqrt(h)[None, :, None] * z[None, None, :])
        return outs, None
USING_FALLBACK = False
try:
    import torch; from chronos import Chronos2Pipeline
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device', dev, torch.cuda.get_device_name(0) if dev == 'cuda' else '')
    PIPE = Chronos2Pipeline.from_pretrained(CFG['model'] if dev == 'cuda' else 'autogluon/chronos-2-small', device_map=dev)
except Exception as e:
    print('!!! Chronos-2 could not be loaded (', type(e).__name__, str(e)[:120], ') -> NAIVE FALLBACK. Results below are NOT Chronos-2.'); PIPE = NaiveQuantilePipeline(); USING_FALLBACK = True
def norm_out(q, n_var):
    a = q.float().cpu().numpy() if hasattr(q, 'float') else np.asarray(q)
    if a.shape[0] != n_var: raise ValueError(f'unexpected output shape {a.shape}')
    if a.shape[1] == PRED and a.shape[2] == len(QL): return a
    if a.shape[1] == len(QL) and a.shape[2] == PRED: return a.transpose(0, 2, 1)
    raise ValueError(f'unexpected output shape {a.shape}')
def forecast(tasks):
    try: q, _ = PIPE.predict_quantiles(tasks, prediction_length=PRED, quantile_levels=QL, batch_size=CFG['batch'])
    except TypeError: q, _ = PIPE.predict_quantiles(tasks, prediction_length=PRED, quantile_levels=QL)
    return [norm_out(x, np.asarray(t['target']).shape[0]) for x, t in zip(q, tasks)]

In [ ]:
# ---------------- per-episode task construction (strict as-of: only asof[:t+1] and the forecast issued at floor(t/6)*6)
def episode_tasks(d, city, profile, seed):
    asof, stale, _ = W.observe(d['X'], profile, seed + 777); Z = asof.shape[1]
    rain = np.nan_to_num(np.nanmean(asof[..., W.VI['rain_mm_h']], 1), nan=0.0); temp = np.nan_to_num(np.nanmean(asof[..., W.VI['temp_c']], 1), nan=28.0)
    tasks, keys = [], []
    for r, t in enumerate(REF_T):
        lo = t + 1 - CTX; pad = max(0, -lo); sl = slice(max(lo, 0), t + 1); i = t // 6; o = t - 6 * i
        fr, ft = d['fs_rain'][i, o:o + PRED], d['fs_temp'][i, o:o + PRED]
        pr = np.r_[np.zeros(pad), rain[sl]]; pt = np.r_[np.full(pad, 28.0), temp[sl]]
        for k, v in enumerate(TGT):
            y = np.r_[np.full((pad, Z), np.nan), asof[sl, :, v]].T.astype(np.float32)
            tasks.append(dict(target=y, past_covariates=dict(rain=pr.astype(np.float32), temp=pt.astype(np.float32)), future_covariates=dict(rain=fr.astype(np.float32), temp=ft.astype(np.float32)))); keys.append((r, k))
    return tasks, keys, asof
# smoke test of the API on 2 tasks
_d = W.load_episode(f'{ROOT}/{MAN.path.iloc[0]}'); _t, _k, _ = episode_tasks(_d, CITY['anchor'], 'F0', 1)
try: _o = forecast(_t[:2]); print('API OK; output', _o[0].shape, 'finite', all(np.isfinite(x).all() for x in _o))
except Exception as e:
    print('!!! Chronos API call failed:', type(e).__name__, e, '\n    -> check the current README signature; switching to naive fallback'); PIPE = NaiveQuantilePipeline(); USING_FALLBACK = True

In [ ]:
# ---------------- throughput benchmark -> episode quota
Z_a = CITY['anchor']['Z']; bt = _t[:48]; t0 = time.time(); forecast(bt); dt = time.time() - t0
if 'torch' in sys.modules and not USING_FALLBACK:
    import torch
    if torch.cuda.is_available(): torch.cuda.synchronize()
t0 = time.time(); forecast(bt); dt = time.time() - t0; sps = len(bt) * Z_a / dt
per_ep_series = len(REF_T) * len(TGT) * Z_a; sec_per_ep = per_ep_series / sps
n_afford = int(CFG['budget_h'] * 3600 / sec_per_ep); print(f'throughput {sps:,.0f} series/s -> {sec_per_ep:.1f} s/episode -> {n_afford} episodes fit in {CFG["budget_h"]} GPU-h' + ('   (FALLBACK speed, not meaningful)' if USING_FALLBACK else ''))
order = ['val', 'test', 'heldout', 'drift', 'train']; avail = {s: min(CFG['max_eps'][s], int((MAN.split == s).sum())) for s in order}; left = n_afford; QUOTA = {}
for s in order: QUOTA[s] = min(avail[s], left) if s != 'train' else min(avail[s], max(left, 0)); left -= QUOTA[s]
QUOTA = {s: max(q, 0) for s, q in QUOTA.items()}; print('episode quota:', QUOTA, '(if train quota is small, raise budget_h or lower val/test sizes)')

In [ ]:
# ---------------- main loop (checkpoint = one npz per episode; re-run to resume)
HIDX = [h - 1 for h in HEV]
def metrics_for(d, asof, fq, r_list):
    """fq: (n_ref, Z, n_target, 72, 3) forecasts; returns array (n_target, len(HEV), 8) of SUMS over (origin, zone):
       [n, |med err|, mean pinball, in-80%-band, persist |err|, persist pinball, seasonal-naive |err|, band width]"""
    from scipy.stats import norm
    out = np.zeros((len(TGT), len(HEV), 8)); z = norm.ppf(QL)
    for ri, t in enumerate(REF_T):
        for k, v in enumerate(TGT):
            y = d['X'][t + 1:t + 1 + PRED, :, v]                                  # TRUTH (clean) future, (72, Z)
            f = fq[ri, :, k]                                                      # (Z, 72, 3)
            last = np.nan_to_num(asof[t, :, v], nan=W.VAR_DEFAULT[v]); sd = np.nanstd(np.diff(asof[max(0, t - 72):t + 1, :, v], axis=0), 0) + 1e-3
            for hi, h in enumerate(HEV):
                yt = y[h - 1]; q = f[:, h - 1]; pin = np.mean([np.maximum(a * (yt - q[:, j]), (a - 1) * (yt - q[:, j])) for j, a in enumerate(QL)], 0)
                pq = last[:, None] + sd[:, None] * np.sqrt(h) * z[None]; ppin = np.mean([np.maximum(a * (yt - pq[:, j]), (a - 1) * (yt - pq[:, j])) for j, a in enumerate(QL)], 0)
                sn = np.nan_to_num(asof[t - 24 + ((h - 1) % 24) + 1, :, v], nan=W.VAR_DEFAULT[v])
                out[k, hi] += [len(yt), np.abs(yt - q[:, 1]).sum(), pin.sum(), ((yt >= q[:, 0]) & (yt <= q[:, 2])).sum(), np.abs(yt - last).sum(), ppin.sum(), np.abs(yt - sn).sum(), (q[:, 2] - q[:, 0]).sum()]
    return out
def run_split(split):
    rows = MAN[MAN.split == split].head(QUOTA[split]); t0 = time.time()
    for r in tqdm(list(rows.itertuples()), desc=f'chronos {split}'):
        path = f'{CF}/{split}/{r.ep_id}.npz'
        if os.path.exists(path): continue
        d = W.load_episode(f'{ROOT}/{r.path}'); tasks, keys, asof = episode_tasks(d, CITY[r.city], r.profile, int(r.seed)); Z = asof.shape[1]
        outs = forecast(tasks); fq = np.zeros((len(REF_T), Z, len(TGT), PRED, 3), np.float32)
        for (ri, k), o in zip(keys, outs): fq[ri, :, k] = o
        feats = np.stack([fq[:, :, :, h - 1, :] for h in HS], 3).astype(np.float16)           # (n_ref, Z, n_target, 3 horizons, 3 quantiles)
        np.savez_compressed(path, feats=feats, t_refresh=REF_T, metr=metrics_for(d, asof, fq, REF_T), profile=r.profile, hazard=r.hazard, using_fallback=USING_FALLBACK)
    return time.time() - t0
TIMES = {s: run_split(s) for s in order}; print({s: round(v / 60, 1) for s, v in TIMES.items()}, 'minutes')

In [ ]:
# ---------------- aggregate metrics
rows = []
for s in order:
    for p in sorted(glob.glob(f'{CF}/{s}/*.npz')):
        z = np.load(p); ep = os.path.basename(p)[:-4]; m = MAN[MAN.ep_id == ep].iloc[0]
        for k, tn in enumerate(W.CH_TARGETS):
            for hi, h in enumerate(HEV):
                a = z['metr'][k, hi]; rows.append(dict(split=s, ep=ep, profile=m.profile, hazard=m.hazard, target=tn, h=h, n=a[0], mae=a[1], pin=a[2], cov=a[3], mae_p=a[4], pin_p=a[5], mae_s=a[6], width=a[7]))
MET = pd.DataFrame(rows); MET.to_csv(f'{OUT}/chronos_metrics.csv', index=False)
ev = MET[MET.split.isin(['val', 'test'])]
agg = lambda df, by: df.groupby(by).apply(lambda g: pd.Series(dict(MAE=g.mae.sum() / g.n.sum(), MAE_persist=g.mae_p.sum() / g.n.sum(), MAE_seasonal=g.mae_s.sum() / g.n.sum(), pinball=g.pin.sum() / g.n.sum(), pinball_persist=g.pin_p.sum() / g.n.sum(),
                               coverage80=g['cov'].sum() / g.n.sum(), width=g.width.sum() / g.n.sum(), n=g.n.sum())))
TH_ = agg(ev, ['target', 'h']); TH_['skill_MAE'] = 1 - TH_.MAE / TH_.MAE_persist; TH_['skill_pinball'] = 1 - TH_.pinball / TH_.pinball_persist
print(('FALLBACK (naive) ' if USING_FALLBACK else 'Chronos-2 '), 'results on val+test, all fault profiles pooled'); display(agg(ev, ['h']).round(4))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4.4)); sk = TH_.skill_pinball.unstack(); im = ax[0].imshow(sk.values, cmap='RdYlGn', vmin=-.5, vmax=.8, aspect='auto'); ax[0].grid(False)
ax[0].set_yticks(range(len(sk))); ax[0].set_yticklabels(sk.index); ax[0].set_xticks(range(len(HEV))); ax[0].set_xticklabels([f'+{h}h' for h in HEV]); plt.colorbar(im, ax=ax[0]); ax[0].set_title('pinball skill vs persistence (1 = perfect, <0 worse)')
for (i, j), v in np.ndenumerate(sk.values): ax[0].text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=7)
for tn in W.CH_TARGETS:
    g = TH_.loc[tn]; ax[1].plot(g.index, g.pinball, '-o', label=tn); ax[1].plot(g.index, g.pinball_persist, ':', c=ax[1].lines[-1].get_color())
ax[1].set_xlabel('horizon (h)'); ax[1].set_ylabel('mean pinball (solid=model, dotted=persistence)'); ax[1].set_title('probabilistic error vs lead'); ax[1].legend(fontsize=7); ax[1].set_yscale('log')
cv = agg(ev, ['profile', 'h']).coverage80.unstack(0); cv.plot(ax=ax[2], marker='o'); ax[2].axhline(.8, c='k', ls='--'); ax[2].set_title('80% interval coverage by fault profile'); ax[2].set_xlabel('horizon (h)'); ax[2].set_ylim(0, 1)
savefig('30_chronos_skill_coverage')

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4)); hz = agg(ev, ['hazard']); hz['skill'] = 1 - hz.pinball / hz.pinball_persist; hz.skill.plot.bar(ax=ax[0], color='tab:green'); ax[0].set_title('pinball skill by hazard (pooled targets/horizons)'); ax[0].axhline(0, c='k')
pf = agg(ev, ['profile']); pf['skill'] = 1 - pf.pinball / pf.pinball_persist; pf.skill.plot.bar(ax=ax[1], color='tab:orange'); ax[1].set_title('skill by fault profile (graceful degradation)'); ax[1].axhline(0, c='k')
ax[2].bar(['measured'], [sps], color='tab:blue'); ax[2].axhline(300, c='r', ls='--'); ax[2].set_ylabel('series / s'); ax[2].set_title('throughput (red: A10G figure from the model card)'); savefig('31_chronos_by_hazard_profile')
# example fan chart
ep = MAN[(MAN.split == 'val') & (MAN.hazard == 'flood')].ep_id; ep = ep.iloc[0] if len(ep) else MAN[MAN.split == 'val'].ep_id.iloc[0]; r = MAN[MAN.ep_id == ep].iloc[0]; z_ = np.load(f'{CF}/val/{ep}.npz')
d = W.load_episode(f'{ROOT}/{r.path}'); Zn = d['X'].shape[1]; zone = int(np.argmax(d['X'][..., W.VI['water_depth_cm']].max(0))); fig, axs = plt.subplots(1, 3, figsize=(17, 3.8))
for a, ri in zip(axs, [2, 6, 10]):
    t = REF_T[ri]; k = W.CH_TARGETS.index('water_depth_cm'); f = z_['feats'][ri, zone, k].astype(float); a.plot(np.arange(t - 48, t + 73), d['X'][t - 48:t + 73, zone, W.VI['water_depth_cm']], 'k', label='truth')
    a.plot(np.array(HS) + t, f[:, 1], 'o-', c='tab:red', label='q50 @ +6/+24/+72h'); a.fill_between(np.array(HS) + t, f[:, 0], f[:, 2], color='tab:red', alpha=.25); a.axvline(t, c='gray', ls='--'); a.set_title(f'{ep} zone {zone} origin t={t}')
axs[0].legend(); savefig('32_chronos_fan_example')
json.dump(dict(model=CFG['model'], fallback=bool(USING_FALLBACK), series_per_s=float(sps), quota=QUOTA, targets=W.CH_TARGETS, horizons=HS, t_refresh=REF_T.tolist(), minutes=TIMES), open(f'{OUT}/chronos_info.json', 'w'), indent=1)
print('episodes with features:', {s: len(glob.glob(f'{CF}/{s}/*.npz')) for s in order}); print('NEXT: NB04 reads chronos_feats/ (any subset of episodes works).')